In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import tensorflow_datasets as tfds

ds_train, ds_test = tfds.load(
    "svhn_cropped",
    split = ["train+extra", "test"],
    as_supervised = True,
)

In [ ]:
tf.random.set_seed(0)

train_data = ds_train.shuffle(buffer_size = 10000, reshuffle_each_iteration = True).batch(32)
test_data = ds_test.batch(32)

Nlabels = 10



In [ ]:
histories = {}
for Nepochs, second in [(6, False), (3, True)]:
    model = keras.models.Sequential()

    # Convolutional layer
    model.add(keras.layers.Input(shape = (32, 32, 3)))
    model.add(keras.layers.Rescaling(1./255))                                  # normalizzazione
    model.add(keras.layers.Conv2D(64, (3,3), padding = 'same'))                # zero-padding
    model.add(keras.layers.Activation("relu"))

    if second:
        # convolutional layer
        model.add(keras.layers.Conv2D(32, (3,3), padding = 'same'))
        model.add(keras.layers.Activation("relu"))

    # Pooling layer
    model.add(keras.layers.MaxPooling2D(pool_size = (2,2)))

    # Fully connected layer
    model.add(keras.layers.Flatten())
    model.add(keras.layers.Dense(Nlabels))
    model.add(keras.layers.Activation("softmax"))

    early_stopping = keras.callbacks.EarlyStopping(
        monitor = 'loss',             # metrica da controllare
        patience = 1,                 # quante epoche tollerare senza miglioramento
        min_delta = 1e-4,
        restore_best_weights = True   # ripristina i pesi migliori
    )

    model.compile(loss = "sparse_categorical_crossentropy",
                  optimizer = "adam",
                  metrics = ["accuracy"])


    training = model.fit(train_data,
                                       epochs = Nepochs,
                                       validation_data = test_data,
                                       callbacks = [early_stopping])
    histories[second] = training.history